![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Transfer Learning Lab — Fine-Tune a Pretrained CNN on Imagenette

In this lab you'll learn **transfer learning**: instead of training a CNN from scratch, you take a model that's already been trained on millions of images (ResNet18, trained on ImageNet) and adapt it to a new, smaller task.

The core idea, in plain words:

1. **Load a pretrained model.** ResNet18 has already learned to recognize edges, textures, shapes, and object parts from 1.2 million ImageNet images.
2. **Freeze the backbone.** We lock all of that learned knowledge in place so it doesn't change.
3. **Replace the classifier head.** We swap out the final layer (which used to predict 1000 ImageNet classes) with a new, small layer that predicts our 10 classes instead.
4. **Train only the new head — fast.** Since the backbone is frozen and never changes, we run it over the dataset **once**, save the resulting features, and train the tiny head on those cached features. This is dramatically faster than re-running the full ResNet18 every single epoch.

**Dataset: Imagenette.** A small, real-photo subset of Large dataset with 10 easy classes (tench, English springer spaniel, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute). Small to download (~94MB), real photos, perfect for a focused fine-tuning exercise.

**Goal:** Fine-tune ResNet18 on Imagenette and see how quickly a frozen pretrained backbone reaches strong accuracy — with a training loop fast enough to run in seconds per epoch.

---
## Roadmap

| Part | What we do |
|---|---|
| Part 1 | Download and load Imagenette |
| Part 2 | Load pretrained ResNet18 and freeze the backbone |
| Part 3 | Replace the classifier head |
| Part 4 | Extract backbone features once (the speed trick) |
| Part 5 | Train the head on cached features (fast) |
| Part 6 | Check the final accuracy |


---
# Part 1 — Download and Load Imagenette

We download Imagenette directly (a single ~94MB file), extract it, and load it with `torchvision.datasets.ImageFolder`, which automatically turns a folder of `class_name/image.jpg` files into a labeled dataset.

We also resize every image to 224x224 and normalize it using ImageNet's mean and standard deviation — this matters because **ResNet18 expects input in exactly the same format it was originally trained on**. If we feed it images in a different size or scale, the pretrained features won't work properly.

No data augmentation here, on purpose — keeping preprocessing simple also lets us cache features later for a big speed boost (more on that in Part 4).

In [ ]:
import os
import requests
import tarfile
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# --- Download Imagenette (small, ~94MB, real photos, 10 classes) ---
DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)

url = 'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz'
tar_path = os.path.join(DATA_DIR, 'imagenette2-160.tgz')
extract_path = os.path.join(DATA_DIR, 'imagenette2-160')

if not os.path.exists(extract_path):
    if not os.path.exists(tar_path):
        print('Downloading Imagenette...')
        response = requests.get(url, stream=True)
        total_size = int(response.headers.get('content-length', 0))
        with open(tar_path, 'wb') as f, tqdm(total=total_size, unit='B', unit_scale=True) as pbar:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                pbar.update(len(chunk))

    print('Extracting...')
    with tarfile.open(tar_path) as tar:
        tar.extractall(DATA_DIR)
    print('Done.')
else:
    print('Imagenette already downloaded and extracted.')

In [ ]:
# --- Load the dataset with ImageFolder ---
# ImageFolder reads a directory structured as: train/class_name/image.jpg
# and automatically assigns a label to each class folder.

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),   # ResNet18 expects 224x224 input
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = torchvision.datasets.ImageFolder(
    root=os.path.join(extract_path, 'train'), transform=transform
)
test_dataset = torchvision.datasets.ImageFolder(
    root=os.path.join(extract_path, 'val'), transform=transform
)

BATCH_SIZE = 128  # larger batch size = fewer, bigger passes = faster on GPU

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)

# ImageFolder uses raw ImageNet WordNet IDs as folder names (e.g. 'n01440764'),
# so we map them to readable names for display purposes.
WNID_TO_NAME = {
    'n01440764': 'tench',
    'n02102040': 'English springer',
    'n02979186': 'cassette player',
    'n03000684': 'chain saw',
    'n03028079': 'church',
    'n03394916': 'French horn',
    'n03417042': 'garbage truck',
    'n03425413': 'gas pump',
    'n03445777': 'golf ball',
    'n03888257': 'parachute',
}

class_names = [WNID_TO_NAME[wnid] for wnid in train_dataset.classes]

print(f'Classes: {class_names}')
print(f'Training images: {len(train_dataset)}')
print(f'Test images:     {len(test_dataset)}')

In [ ]:
# --- Quick look at one sample image per class ---
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

# Find the first image index belonging to each class
shown_classes = set()
sample_indices = []
for idx, (_, label) in enumerate(train_dataset.samples):
    if label not in shown_classes:
        sample_indices.append(idx)
        shown_classes.add(label)
    if len(shown_classes) == len(class_names):
        break

# Sort so classes appear in order (0 to 9) across the grid
sample_indices.sort(key=lambda idx: train_dataset.samples[idx][1])

for ax, idx in zip(axes.flat, sample_indices):
    img, label = train_dataset[idx]
    img = img.permute(1, 2, 0).numpy()
    img = img * IMAGENET_STD + IMAGENET_MEAN  # undo normalization just for display
    img = img.clip(0, 1)
    ax.imshow(img)
    ax.set_title(class_names[label], fontsize=10)
    ax.axis('off')

fig.suptitle('Imagenette — One Sample per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 2 — Load Pretrained ResNet18 and Freeze the Backbone

This is the heart of transfer learning. Two steps:

**Step 1 — Load ResNet18 with its pretrained ImageNet weights.**
One line of code gives us a model that already knows how to "see" — it has already learned useful visual features from 1.2 million images.

**Step 2 — Freeze every parameter.**
"Freezing" means telling PyTorch *"don't update these weights during training."* We do this by setting `requires_grad = False` on every parameter. The model's existing knowledge stays exactly as it is — only the new layer we add in Part 3 will actually learn anything.

**Why freeze instead of training the whole thing?**
- Much faster to train
- Needs far less data — we don't have millions of images, just a few thousand
- Keeps the valuable pretrained knowledge intact instead of risking it being overwritten with too little new data

In [ ]:
# --- Load ResNet18 with pretrained ImageNet weights ---
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)

print('Loaded ResNet18 pretrained on ImageNet.')
print(f'Original final layer (fc): {model.fc}')

In [ ]:
# --- Freeze every parameter in the model ---
for param in model.parameters():
    param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}  (should be 0 — everything is frozen)')

---
# Part 3 — Replace the Classifier Head

ResNet18's original final layer was built to output **1000** scores, one per ImageNet class. We don't want that — we want **10** scores, one per Imagenette class.

So we replace `model.fc` with a brand new `nn.Linear` layer. Because this layer is freshly created, PyTorch automatically sets its `requires_grad` to `True` — meaning **this is the only part of the model that will actually train.**

In [ ]:
# Find out how many features the backbone outputs, right before the final layer
num_features = model.fc.in_features
print(f'ResNet18 backbone outputs {num_features} features before the final layer.')

NUM_CLASSES = len(class_names)  # 10 for Imagenette

# Replace the classifier head — this new layer is trainable by default
model.fc = nn.Linear(num_features, NUM_CLASSES)

model = model.to(device)

print(f'\nNew classifier head: {model.fc}')

In [ ]:
# --- Confirm only the new head is trainable ---
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'\nOnly {100 * trainable_params / total_params:.3f}% of the model will actually be trained.')

---
# Part 4 — Extract Backbone Features Once (the speed trick)

Here's the key insight: **the frozen backbone never changes.** So instead of running every image through the full ResNet18 backbone again and again, once per epoch, we can run it **exactly once**, save the resulting feature vector for every image, and reuse those saved features for every epoch afterward.

We split the model into two pieces:
- `feature_extractor` — everything except the final layer (the frozen part)
- `model.fc` — the new classifier head (the trainable part)

Each image becomes a single vector of 512 numbers (ResNet18's feature size) after passing through the frozen backbone. We do this once for the whole training and test sets, and store the results.

**This is the single biggest speedup available for this setup** — turning "run ResNet18 over 9,469 images every epoch" into "run ResNet18 over 9,469 images one time total."

*Note: this trick relies on having no data augmentation — since we kept preprocessing simple in Part 1, every image always produces the same features, so caching them is safe.*

In [ ]:
# --- Split the model: everything except model.fc is the frozen feature extractor ---
feature_extractor = nn.Sequential(*list(model.children())[:-1])  # drops model.fc
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

def extract_features(loader):
    all_features = []
    all_labels = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            feats = feature_extractor(images)   # shape: (batch, 512, 1, 1)
            feats = feats.flatten(1)             # shape: (batch, 512)
            all_features.append(feats.cpu())
            all_labels.append(labels)
    return torch.cat(all_features), torch.cat(all_labels)

print('Extracting training features (runs ResNet18 once over the training set)...')
train_features, train_labels = extract_features(train_loader)

print('Extracting test features...')
test_features, test_labels = extract_features(test_loader)

print(f'\nTrain features shape: {train_features.shape}')
print(f'Test features shape:  {test_features.shape}')

---
# Part 5 — Train the Head on Cached Features (fast)

Now training is simple and fast: the cached features are just plain tensors, so we wrap them in a `TensorDataset` and train the small classifier head on them directly — no images, no backbone, no GPU-heavy convolutions involved at all in this loop.

Each epoch now only does a forward/backward pass through one `Linear(512, 10)` layer on cached vectors, so this should take seconds rather than minutes, even on a free-tier GPU.

In [ ]:
# Wrap cached features into simple datasets/loaders
feat_train_dataset = TensorDataset(train_features, train_labels)
feat_test_dataset  = TensorDataset(test_features, test_labels)

FEAT_BATCH_SIZE = 256
feat_train_loader = DataLoader(feat_train_dataset, batch_size=FEAT_BATCH_SIZE, shuffle=True)
feat_test_loader  = DataLoader(feat_test_dataset, batch_size=FEAT_BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

NUM_EPOCHS = 5
head = model.fc.to(device)

In [ ]:
train_losses = []
val_accuracies = []

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    head.train()
    running_loss = 0.0
    num_batches = 0

    for feats, labels in feat_train_loader:
        feats, labels = feats.to(device), labels.to(device)

        outputs = head(feats)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1

    avg_loss = running_loss / num_batches
    train_losses.append(avg_loss)

    # --- Validation ---
    head.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for feats, labels in feat_test_loader:
            feats, labels = feats.to(device), labels.to(device)
            outputs = head(feats)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100.0 * correct / total
    val_accuracies.append(val_acc)

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]  Loss: {avg_loss:.4f}  Val Acc: {val_acc:.2f}%')

## Plot Training Curves

A quick look at how loss and accuracy moved across epochs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_range, train_losses, 'b-o')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, val_accuracies, 'g-o')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Validation Accuracy')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# Part 6 — Final Accuracy

A clean final check on the test set, using the same cached features.

In [ ]:
head.eval()
correct = 0
total = 0

with torch.no_grad():
    for feats, labels in feat_test_loader:
        feats, labels = feats.to(device), labels.to(device)
        outputs = head(feats)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100.0 * correct / total
print(f'Final Test Accuracy: {test_acc:.2f}%')

In [ ]:
# --- Test on random images and visualize predictions ---
import random

model.fc = head  # make sure the trained head is attached back to the full model
model.eval()

NUM_SAMPLES = 10
sample_indices = random.sample(range(len(test_dataset)), NUM_SAMPLES)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))

with torch.no_grad():
    for ax, idx in zip(axes.flat, sample_indices):
        img, true_label = test_dataset[idx]

        # Run the actual full model (backbone + head) on this one image
        input_tensor = img.unsqueeze(0).to(device)  # add batch dimension
        output = model(input_tensor)
        pred_label = output.argmax(dim=1).item()

        # Undo normalization just for display
        display_img = img.permute(1, 2, 0).numpy()
        display_img = display_img * IMAGENET_STD + IMAGENET_MEAN
        display_img = display_img.clip(0, 1)

        is_correct = (pred_label == true_label)
        color = 'green' if is_correct else 'red'

        ax.imshow(display_img)
        ax.set_title(
            f'True: {class_names[true_label]}\nPred: {class_names[pred_label]}',
            fontsize=10, color=color
        )
        ax.axis('off')

fig.suptitle('Model Predictions on Random Test Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Conclusion

You just performed transfer learning end-to-end, with a fast training setup:

| Step | What it does |
|---|---|
| Load pretrained ResNet18 | Reuse knowledge learned from 1.2 million ImageNet images |
| Freeze all parameters | Lock that knowledge in place, nothing gets overwritten |
| Replace `model.fc` | Swap the 1000-class output for our 10-class output |
| Cache backbone features once | Avoid re-running the expensive frozen backbone every epoch |
| Train only the head on cached features | Each epoch becomes a fast pass through one small layer |

**The big takeaway:** when the backbone is frozen, it never needs to be recomputed — caching its output turns transfer learning into a quick, lightweight training process, even on modest hardware.

---


### Contributed by Hussain Alyafei